# Sort Batches by Reproductive Annotation

Reads every JSON file from `prompts_batches_20_reverif/batch_XX/` and copies it into
`sorted_batches_reprod_annot/batch_XX/` under one of two sub-folders:

| Sub-folder | Condition |
|---|---|
| `with_reprod_annot/` | `corrected_text` contains at least one of `[Ri `, `[N_R_i `, `[F_R_i ` |
| `without_reprod_annot/` | none of the three patterns found in `corrected_text` |

The three patterns always appear at the **start** of an annotation bracket, e.g.
`[Ri = 4 eggs per year]`, `[N_R_i = 3-5 eggs per clutch]`, `[F_R_i = about 1 clutch each year]`.

**Run Cell 1 (config) then Cell 2 (sort). Cell 3 prints a per-batch summary.**

In [1]:
import json, re, shutil
from pathlib import Path

# -- Configuration ------------------------------------------------------------
SOURCE_ROOT = Path("../prompts_batches_20_reverif")
DEST_ROOT   = Path("../sorted_batches_20_reprod_annot")

N_BATCHES = 20

# Patterns that mark a reproductive annotation (case-sensitive, bracket-start)
REPROD_PATTERNS = re.compile(r'\[(?:Ri |N_R_i |F_R_i )')

# -----------------------------------------------------------------------------
print(f"Source : {SOURCE_ROOT.resolve()}")
print(f"Dest   : {DEST_ROOT.resolve()}")
print(f"Batches: {N_BATCHES}")
print(f"Pattern: {REPROD_PATTERNS.pattern}")

Source : /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/prompts_batches_20_reverif
Dest   : /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/sorted_batches_20_reprod_annot
Batches: 20
Pattern: \[(?:Ri |N_R_i |F_R_i )


## Cell 2 — Sort
Iterates over all 20 batch folders, checks each JSON file's `corrected_text` field,
and **copies** (does not move) it to the matching sub-folder in `sorted_batches_reprod_annot`.

In [2]:
total_with = total_without = total_skipped = 0

for batch_num in range(1, N_BATCHES + 1):
    batch_name = f"batch_{batch_num:02d}"
    src_dir  = SOURCE_ROOT / batch_name
    dst_with = DEST_ROOT / batch_name / "with_reprod_annot"
    dst_without = DEST_ROOT / batch_name / "without_reprod_annot"

    if not src_dir.exists():
        print(f"[WARN] {src_dir} not found — skipping")
        continue

    dst_with.mkdir(parents=True, exist_ok=True)
    dst_without.mkdir(parents=True, exist_ok=True)

    n_with = n_without = n_skip = 0

    for fp in sorted(src_dir.glob("*.json")):
        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
        except json.JSONDecodeError:
            print(f"  [PARSE-ERROR] {fp.name}")
            n_skip += 1
            continue

        text = data.get("corrected_text", "")
        if not text:
            # no text field — put in without
            shutil.copy2(fp, dst_without / fp.name)
            n_without += 1
            continue

        if REPROD_PATTERNS.search(text):
            shutil.copy2(fp, dst_with / fp.name)
            n_with += 1
        else:
            shutil.copy2(fp, dst_without / fp.name)
            n_without += 1

    print(f"{batch_name}: {n_with} with_reprod_annot | {n_without} without_reprod_annot"
          + (f" | {n_skip} skipped" if n_skip else ""))
    total_with    += n_with
    total_without += n_without
    total_skipped += n_skip

print(f"\nTotal: {total_with} with | {total_without} without | {total_skipped} skipped")

batch_01: 318 with_reprod_annot | 52 without_reprod_annot
batch_02: 313 with_reprod_annot | 57 without_reprod_annot
batch_03: 323 with_reprod_annot | 47 without_reprod_annot
batch_04: 300 with_reprod_annot | 70 without_reprod_annot
batch_05: 325 with_reprod_annot | 45 without_reprod_annot
batch_06: 303 with_reprod_annot | 67 without_reprod_annot
batch_07: 314 with_reprod_annot | 56 without_reprod_annot
batch_08: 329 with_reprod_annot | 41 without_reprod_annot
batch_09: 307 with_reprod_annot | 63 without_reprod_annot
batch_10: 306 with_reprod_annot | 64 without_reprod_annot
batch_11: 320 with_reprod_annot | 50 without_reprod_annot
batch_12: 328 with_reprod_annot | 42 without_reprod_annot
batch_13: 296 with_reprod_annot | 74 without_reprod_annot
batch_14: 321 with_reprod_annot | 49 without_reprod_annot
batch_15: 302 with_reprod_annot | 68 without_reprod_annot
batch_16: 322 with_reprod_annot | 48 without_reprod_annot
batch_17: 304 with_reprod_annot | 66 without_reprod_annot
batch_18: 306 

## Cell 3 — Summary
Reads the destination folders and prints counts to verify the sort.

In [4]:
grand_with = grand_without = 0

print(f"{'Batch':<12} {'with_reprod_annot':>18} {'without_reprod_annot':>22}")
print("-" * 55)

for batch_num in range(1, N_BATCHES + 1):
    batch_name = f"batch_{batch_num:02d}"
    n_with    = len(list((DEST_ROOT / batch_name / "with_reprod_annot").glob("*.json")))
    n_without = len(list((DEST_ROOT / batch_name / "without_reprod_annot").glob("*.json")))
    grand_with    += n_with
    grand_without += n_without
    print(f"{batch_name:<12} {n_with:>18} {n_without:>22}")

print("-" * 55)
print(f"{'TOTAL':<12} {grand_with:>18} {grand_without:>22}")
print(f"\nGrand total: {grand_with + grand_without} files")

Batch         with_reprod_annot   without_reprod_annot
-------------------------------------------------------
batch_01                    318                     52
batch_02                    313                     57
batch_03                    323                     47
batch_04                    300                     70
batch_05                    325                     45
batch_06                    303                     67
batch_07                    314                     56
batch_08                    329                     41
batch_09                    307                     63
batch_10                    306                     64
batch_11                    320                     50
batch_12                    328                     42
batch_13                    296                     74
batch_14                    321                     49
batch_15                    302                     68
batch_16                    322                     48
batch_17 

## Cell 4 — Collect all without_reprod_annot into one folder
Copies every file from each `without_reprod_annot/` sub-folder into a single flat
folder `all_without_reprod_annot/`, appending `_batch_XX` to each filename so the
origin batch is always traceable.

Example: `0001_Abatus_cordatus_abj.json` from batch_01 becomes
`0001_Abatus_cordatus_abj_batch_01.json`.

In [5]:
ALL_WITHOUT_DIR = DEST_ROOT.parent / "all_without_reprod_annot"
ALL_WITHOUT_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
for batch_num in range(1, N_BATCHES + 1):
    batch_name = f"batch_{batch_num:02d}"
    src_dir = DEST_ROOT / batch_name / "without_reprod_annot"
    if not src_dir.exists():
        print(f"[WARN] {src_dir} not found — skipping")
        continue
    for fp in sorted(src_dir.glob("*.json")):
        dest_name = f"{fp.stem}_{batch_name}{fp.suffix}"
        shutil.copy2(fp, ALL_WITHOUT_DIR / dest_name)
        copied += 1

print(f"Copied {copied} files to {ALL_WITHOUT_DIR.resolve()}")

Copied 1094 files to /Users/somaya/Documents/Phd/github_version/version_2/deb-data-pipeline-github/data_verification/all_without_reprod_annot
